# BTS flight matching investigation

Compare BTS flight records with ADS-B positions for one day. A match is evaluated for the full flight sequence of each aircraft (ICAO).

## Setup

In [1]:
from datetime import date, datetime
from pathlib import Path
import sys

import polars as pl

project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src").is_dir()
)
sys.path.insert(0, str(project_root / "src"))

from data_engineering.adsb.read_adsb import read_adsb
from data_engineering.bts.read_bts_ontime_to_flights import get_bts_flights_for_day
from flights.flights_match_adsb import add_adsb_match_column

target_date = date(2026, 3, 1)
adsb_source = "adsbx"

Loading airports from /Volumes/T2-SSD/planequery/data/raw/ourairports/airports.csv


## Load BTS and ADS-B data

In [2]:
df_bts_flights = get_bts_flights_for_day(target_date)
df_adsb = read_adsb(target_date, source=adsb_source)

print(f"BTS flights: {df_bts_flights.height:,}")
print(f"ADS-B messages: {df_adsb.height:,}")

6 overlapping flights found for 2026-03-01
BTS flights: 17,609
ADS-B messages: 92,759,895


## Match flight sequences

`adsb_matched` is repeated on each flight for the ICAO so the original flight rows remain available for inspection. It is `null` when the ADS-B match is indeterminate.

In [3]:
start_dt = datetime(target_date.year, target_date.month, target_date.day)
df_bts_with_match = add_adsb_match_column(df_bts_flights, df_adsb, start_dt)
df_matching_flights = df_bts_with_match.filter(pl.col("adsb_matched") == True)
df_unmatched_flights = df_bts_with_match.filter(pl.col("adsb_matched") == False)
df_unknown_flights = df_bts_with_match.filter(pl.col("adsb_matched").is_null())

## Match coverage

In [4]:
df_icao_matches = df_bts_with_match.group_by("icao").agg(
    pl.col("adsb_matched").first()
)

match_summary = pl.DataFrame({
    "metric": [
        "BTS flights",
        "Matched BTS flights",
        "Unmatched BTS flights",
        "Unknown BTS flights",
        "BTS ICAOs",
        "Matched ICAOs",
        "Unmatched ICAOs",
        "Unknown ICAOs",
    ],
    "total": [
        df_bts_with_match.height,
        df_bts_with_match.height,
        df_bts_with_match.height,
        df_bts_with_match.height,
        df_icao_matches.height,
        df_icao_matches.height,
        df_icao_matches.height,
        df_icao_matches.height,
    ],
    "count": [
        df_bts_with_match.height,
        df_matching_flights.height,
        df_unmatched_flights.height,
        df_unknown_flights.height,
        df_icao_matches.height,
        df_icao_matches.filter(pl.col("adsb_matched") == True).height,
        df_icao_matches.filter(pl.col("adsb_matched") == False).height,
        df_icao_matches.filter(pl.col("adsb_matched").is_null()).height,
    ],
})

match_summary.with_columns(
    (pl.col("count") / pl.col("total") * 100).alias("percent_of_total")
)

metric,total,count,percent_of_total
str,i64,i64,f64
"""BTS flights""",17609,17609,100.0
"""Matched BTS flights""",17609,4218,23.95366
"""Unmatched BTS flights""",17609,7780,44.181952
"""Unknown BTS flights""",17609,5611,31.864388
"""BTS ICAOs""",4718,4718,100.0
"""Matched ICAOs""",4718,1017,21.555744
"""Unmatched ICAOs""",4718,2341,49.618482
"""Unknown ICAOs""",4718,1360,28.825774


## Investigate unmatched aircraft

In [7]:
df_unmatched_icaos = (
    df_icao_matches
    .filter(pl.col("adsb_matched") == False)
    .select("icao")
    .sort("icao")
)

df_unmatched_icaos

icao
str
"""a009a4"""
"""a00d5b"""
"""a01037"""
"""a01097"""
"""a011ac"""
…
"""ade9c0"""
"""adea26"""
"""adeca5"""


In [9]:
# Change this ICAO to inspect a different matched or unmatched sequence.
icao_to_inspect = "a009a4"

df_bts_flights.filter(pl.col("icao") == icao_to_inspect).sort("takeoff_time")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""a009a4""",null,"""N101NN""",2026-03-01 11:43:00,"""KJFK""",2026-03-01 17:26:00,"""KLAX""",false,false,false,false,null,null,null,"""A3"""
"""a009a4""",null,"""N101NN""",2026-03-01 20:37:00,"""KLAX""",2026-03-02 01:23:00,"""KBOS""",false,false,false,false,null,null,null,"""A3"""


In [11]:
from flights.evaluation.evaluation import main

outputs = main("algorithm", "bts", adsb_src="adsbx", test_dates=target_date)

6 overlapping flights found for 2026-03-01
Stats for algorithm:bts: {'true_positive': 3616, 'false_positive': 0, 'false_negative': 0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'takeoff_airport_ident_match_pct': 1.0, 'landing_airport_ident_match_pct': 1.0, 'airport_ident_match_pct': 1.0}
Stats CSV saved to /Volumes/T2-SSD/planequery/data/experiments/comparison/2026-07-24_22-11_224ef38_flights_evaluation.csv
